# 📘 Notebook 06: Building Snake

### Course: *From Random Moves to Winning Agents: Reinforcement Learning with Games*
**Instructor:** Ioannis Tsioulis

*Module B: Learning with a Table · Notebook 2 of 3 in this module · (6 of 18 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- List the design decisions behind every reinforcement learning environment
- Explain why the **full state** of a game is usually too large for a table, with a count
- Design a compact **observation** from features, and say what information it loses
- Design a **reward** that states the goal without dictating the method
- Implement a complete **Gymnasium environment**: spaces, `reset`, `step`, `render`
- Measure a **random** and a **rule-based** baseline, and record videos of both

> **Prerequisites:** Notebook 02 (the `reset` / `step` / `render` interface, `terminated` and `truncated`) and Python classes.
>
> 🔭 **Why this notebook matters later:** up to now the environments were given. In practice, turning a real problem into states, actions and rewards is most of the work, and it decides whether learning succeeds. The environment built here is the one the agent learns in Notebook 07, and the weakness we build into its observation is the reason Module C is needed.

> 📦 **Libraries used in this notebook.** **Gymnasium** supplies the base class and the tools for describing observations and actions, **imageio** makes the videos. Every cell finishes within a few seconds.

In [ ]:
%pip install -q "gymnasium>=1.3" imageio imageio-ffmpeg

In [ ]:
import base64
import numpy as np
import matplotlib.pyplot as plt
import imageio
import gymnasium as gym
from IPython.display import HTML, display


def show_video(frames, fps=10, width=300):
    """Play a list of pictures (RGB arrays) as a video inside the notebook."""
    imageio.mimsave("episode.mp4", frames, fps=fps, macro_block_size=1)
    data = base64.b64encode(open("episode.mp4", "rb").read()).decode()
    display(HTML(f'<video width="{width}" controls autoplay loop src="data:video/mp4;base64,{data}"></video>'))

## 1. Five questions

### Intuition first
Snake is the game from old mobile phones: a snake moves around a board, eats food, grows by one segment each time, and dies when it runs into a wall or into itself. A person understands these rules in ten seconds. An agent understands nothing until we have translated the game into the language of Notebook 02.

### Formal definition
Building an environment means answering five questions:

| Question | In the language of RL |
|---|---|
| What can the agent do? | the **action space** |
| What does the agent see? | the **observation space** |
| What are we asking for? | the **reward** |
| When is a game over? | `terminated` and `truncated` |
| How do we watch it? | `render` |

### Why it matters
None of these has one correct answer. Each is a design decision with consequences for learning, and we take them one at a time before writing any code.

## 2. Decision one: the actions

The obvious choice is four actions: up, right, down, left. It has a flaw. A snake moving right cannot turn round on the spot, so "left" would be either illegal or instantly fatal, and the agent would have to learn that separately for every direction.

We use three **relative** actions: **0 = straight on, 1 = turn right, 2 = turn left**, as seen from the snake's head. Every action is always legal, and "turn right" means the same thing wherever the snake is heading.

## 3. Decision two: what the agent sees

### Intuition first
The complete situation of the game is the position of every segment of the snake, plus the position of the food. That is a proper Markov state. Could we give each such situation a row in a Q-table?

Count them. The cell below counts the ways a snake of a given length can lie on a 10 × 10 board, by trying every way of adding one segment after another.

In [ ]:
def count_snakes(length, size=10):
    """Number of different ways a snake of this length can lie on the board."""
    def extend(cells):
        if len(cells) == length:
            return 1
        x, y = cells[-1]
        total = 0
        for dx, dy in ((0, -1), (1, 0), (0, 1), (-1, 0)):
            nxt = (x + dx, y + dy)
            if 0 <= nxt[0] < size and 0 <= nxt[1] < size and nxt not in cells:
                total += extend(cells + [nxt])
        return total
    return sum(extend([(x, y)]) for x in range(size) for y in range(size))


for length in (3, 5, 7, 9, 11):
    shapes = count_snakes(length)
    print(f"snake of length {length:>2}: {shapes:>9,} positions  x {100 - length} places for the food = {shapes * (100 - length):>13,} states")

Each extra pair of segments multiplies the count by about six. A snake of length 11 already has over a hundred million states, and a good game reaches length 30 or more. No table can hold that, and no agent could visit each state often enough to learn its value.

### A compact observation
So the agent will **not** see the whole board. We describe each situation by 11 yes/no answers:

| Features | Question |
|---|---|
| 3 | Is there **danger** (a wall or my own body) directly **ahead**, to my **right**, to my **left**? |
| 4 | Am I moving **up**, **right**, **down**, **left**? (exactly one is true) |
| 4 | Is the food to the **left** of my head, to the **right**, **above**, **below**? |

Eleven bits give at most $2^{11} = 2048$ combinations, and many of those cannot occur. That fits a table easily.

### Why it matters
This is a trade. The agent gains a state space small enough to learn in. It loses information: it sees danger only in the three cells next to its head, and knows nothing about where the rest of its body lies. Two very different boards can produce the same 11 answers. The observation is **no longer a Markov state**, and we will see what that costs.

## 4. Decision three: the reward

| Event | Reward |
|---|---|
| eating the food | +10 |
| dying | −10 |
| any other step | 0 |

This states what we want (eat, do not die) and nothing about how. We resist the temptation to pay the snake for moving towards the food, or for staying alive. Notebook 07 tries such rewards and shows what goes wrong.

## 5. Decision four: when the game ends

- **`terminated`**: the snake has died, or it fills the whole board (a perfect game).
- **`truncated`**: the snake has gone 200 steps without eating. Without this limit, a snake that has learned to circle safely would play for ever and the training loop would never finish.

## 6. The environment

Here is the complete class. It inherits from `gym.Env`, declares its two spaces in `__init__`, and implements `reset`, `step` and `render` as in Notebook 02. Read `step` slowly: it is the rule book of the game.

In [ ]:
class SnakeEnv(gym.Env):
    """Snake on a square grid. The agent sees 11 yes/no features, not the whole board."""

    metadata = {"render_modes": ["rgb_array"], "render_fps": 10}
    DIRECTIONS = [(0, -1), (1, 0), (0, 1), (-1, 0)]        # up, right, down, left, as (dx, dy)

    def __init__(self, size=10, render_mode="rgb_array"):
        self.size = size
        self.render_mode = render_mode
        self.observation_space = gym.spaces.MultiBinary(11)
        self.action_space = gym.spaces.Discrete(3)          # 0 straight on, 1 turn right, 2 turn left

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)                            # prepares self.np_random
        middle = self.size // 2
        self.snake = [(middle, middle), (middle - 1, middle), (middle - 2, middle)]     # head first
        self.direction = 1                                  # moving right
        self.score = 0
        self.steps_since_food = 0
        self._place_food()
        return self._observation(), {"score": self.score}

    def _place_food(self):
        free = [(x, y) for x in range(self.size) for y in range(self.size) if (x, y) not in self.snake]
        self.food = free[int(self.np_random.integers(len(free)))]

    def _cell_ahead(self, direction):
        dx, dy = self.DIRECTIONS[direction]
        return (self.snake[0][0] + dx, self.snake[0][1] + dy)

    def _is_deadly(self, cell):
        x, y = cell
        return not (0 <= x < self.size and 0 <= y < self.size) or cell in self.snake

    def _observation(self):
        d, head, food = self.direction, self.snake[0], self.food
        features = [
            self._is_deadly(self._cell_ahead(d)),               # danger straight ahead
            self._is_deadly(self._cell_ahead((d + 1) % 4)),     # danger on my right
            self._is_deadly(self._cell_ahead((d - 1) % 4)),     # danger on my left
            d == 0, d == 1, d == 2, d == 3,                     # moving up, right, down, left
            food[0] < head[0], food[0] > head[0],               # food is to the left, to the right
            food[1] < head[1], food[1] > head[1],               # food is above, below
        ]
        return np.array(features, dtype=np.int8)

    def step(self, action):
        self.direction = (self.direction + (0, 1, -1)[action]) % 4
        new_head = self._cell_ahead(self.direction)
        self.steps_since_food += 1
        if self._is_deadly(new_head):                           # hit a wall or itself
            cause = "body" if new_head in self.snake else "wall"
            return self._observation(), -10.0, True, False, {"score": self.score, "death": cause}
        self.snake.insert(0, new_head)
        reward, terminated = 0.0, False
        if new_head == self.food:
            self.score += 1
            self.steps_since_food = 0
            reward = 10.0
            if len(self.snake) == self.size ** 2:               # the board is full: a perfect game
                terminated = True
            else:
                self._place_food()
        else:
            self.snake.pop()                                    # no food: the tail moves on
        truncated = self.steps_since_food >= 2 * self.size ** 2     # going round in circles
        return self._observation(), reward, terminated, truncated, {"score": self.score}

    def render(self, cell=24):
        image = np.full((self.size * cell, self.size * cell, 3), 25, dtype=np.uint8)

        def paint(position, colour, margin=2):
            x, y = position
            image[y * cell + margin:(y + 1) * cell - margin, x * cell + margin:(x + 1) * cell - margin] = colour

        paint(self.food, (220, 60, 60))
        for part in self.snake[1:]:
            paint(part, (70, 170, 90))
        paint(self.snake[0], (160, 235, 130), margin=1)
        return image

Gymnasium can check that a class follows its conventions: that observations have the declared shape, that `reset` accepts a seed, that `step` returns five values of the right types. It is worth running on every environment you write.

In [ ]:
from gymnasium.utils.env_checker import check_env

check_env(SnakeEnv(), skip_render_check=True)
print("The environment follows the Gymnasium interface.")

## 7. A first look

In [ ]:
FEATURES = ["danger ahead", "danger right", "danger left",
            "moving up", "moving right", "moving down", "moving left",
            "food left", "food right", "food above", "food below"]


def describe(observation):
    return ", ".join(name for name, bit in zip(FEATURES, observation) if bit)


env = SnakeEnv()
observation, info = env.reset(seed=0)
print("observation:", observation)
print("in words:   ", describe(observation))

plt.figure(figsize=(3, 3))
plt.imshow(env.render())
plt.axis("off")
plt.show()

The light green square is the head, the darker ones the body, the red one the food. Check the words against the picture.

Now a few steps by hand: turn left, go straight, go straight.

In [ ]:
for action, name in ((2, "turn left"), (0, "straight"), (0, "straight")):
    observation, reward, terminated, truncated, info = env.step(action)
    print(f"{name:10s} -> reward {reward:5.1f} | {describe(observation)}")

## 8. A random snake

Before any learning we need a baseline: how well does an agent do that knows nothing?

In [ ]:
def play(env, policy, seed=None, record=False):
    """One game. Returns the score, the number of steps, how the game ended, and the frames."""
    observation, info = env.reset(seed=seed)
    frames, steps = [], 0
    terminated = truncated = False
    while not (terminated or truncated):
        if record:
            frames.append(env.render())
        observation, reward, terminated, truncated, info = env.step(policy(observation))
        steps += 1
    if record:
        frames.append(env.render())
    ending = info.get("death", "time limit" if truncated else "board full")
    return info["score"], steps, ending, frames


rng = np.random.default_rng(0)


def random_policy(observation):
    return int(rng.integers(3))


def measure(env, policy, games=300):
    results = [play(env, policy, seed=game)[:2] for game in range(games)]
    scores = [score for score, steps in results]
    return float(np.mean(scores)), int(np.max(scores)), float(np.mean([steps for score, steps in results]))


average, best, length = measure(env, random_policy)
print(f"random snake: average score {average:.2f}, best score {best}, average game length {length:.0f} steps")

score, steps, ending, frames = play(env, random_policy, seed=1, record=True)
show_video(frames)

The random snake dies within a couple of dozen steps and almost never finds the food. Any learning agent must beat this.

## 9. A snake with hand-written rules

A second baseline tells us more: how well can **we** play, using only the 11 features the agent will have? The rule is the one most people would think of: go towards the food unless that is dangerous, and otherwise go wherever it is safe.

In [ ]:
def rule_based(observation):
    """Go towards the food if that is safe. Otherwise go anywhere that is safe."""
    danger = observation[0:3]                               # for the actions straight, right, left
    heading = int(np.argmax(observation[3:7]))
    wanted = []                                             # directions that bring the head closer to the food
    if observation[7]:
        wanted.append(3)                                    # food to the left: moving left helps
    if observation[8]:
        wanted.append(1)
    if observation[9]:
        wanted.append(0)
    if observation[10]:
        wanted.append(2)
    for action, turn in ((0, 0), (1, 1), (2, -1)):
        if (heading + turn) % 4 in wanted and not danger[action]:
            return action
    for action in (0, 1, 2):
        if not danger[action]:
            return action
    return 0


average, best, length = measure(env, rule_based)
print(f"rule-based snake: average score {average:.1f}, best score {best}, average game length {length:.0f} steps")

score, steps, ending, frames = play(env, rule_based, seed=1, record=True)
print(f"this game: score {score}, ended by: {ending}")
show_video(frames, fps=15)

Twelve lines of rules take the average score from almost nothing to well over ten. Watch the end of the video, though. The snake does not die against a wall. It **traps itself**: it turns into a pocket formed by its own body, and by the time danger shows up in the three cells around its head, there is no way out.

That is the price of the compact observation from Section 3. The rules cannot avoid a trap they cannot see, and neither will a learning agent that sees the same 11 bits. Keep this in mind for the next notebook: when the learned snake dies in the same way, the fault will lie with what we let it **see**, not with the learning algorithm.

> ⚠️ **Common pitfalls**
>
> - **An observation that hides what matters.** If two situations that need different actions look identical to the agent, no algorithm can tell them apart.
> - **An observation that is too detailed.** The opposite mistake: so many states that none is visited twice, and nothing learned in one carries over to another.
> - **No time limit.** An agent that discovers a safe loop will stay in it. Always give an episode a way to end.
> - **Rewarding the method.** Pay for the result. Section 4 kept the reward to two events for that reason.

---
## ✏️ Exercises

The exercises use `SnakeEnv`, `describe`, `play`, `measure`, `random_policy`, `rule_based` and `show_video` from above.

### Exercise 1 (Predict the observation)
The cell below places the snake and the food by hand. Before running it, write down which of the 11 features you expect to be true. Then run it and compare.

In [ ]:
env = SnakeEnv()
env.reset(seed=0)
env.snake = [(9, 3), (8, 3), (7, 3), (7, 4)]      # head first; x counts from the left, y from the top
env.direction = 1                                  # moving right
env.food = (2, 8)

plt.figure(figsize=(3, 3))
plt.imshow(env.render())
plt.axis("off")
plt.show()
# Your prediction, then: print(describe(env._observation()))


<details><summary>💡 Show solution</summary>

```python
env = SnakeEnv()
env.reset(seed=0)
env.snake = [(9, 3), (8, 3), (7, 3), (7, 4)]
env.direction = 1
env.food = (2, 8)
print(describe(env._observation()))
```

*The head is in the last column and moving right, so the wall is directly ahead. Right of the head (as the snake sees it) is the cell below, which is free, and so is the cell above. The food lies to the left and below.*
</details>

### Exercise 2 (How do the games end?)
For the random snake and for the rule-based snake, play 500 games each and count how many ended against a **wall**, against the snake's own **body**, and by the **time limit**. What is the main cause of death for each?

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
for name, policy in (("random", random_policy), ("rule-based", rule_based)):
    endings = {}
    for game in range(500):
        ending = play(env, policy, seed=game)[2]
        endings[ending] = endings.get(ending, 0) + 1
    print(f"{name:11s}", ", ".join(f"{ending}: {100 * count / 500:.0f}%" for ending, count in sorted(endings.items())))
```

*The random snake mostly runs into a wall, long before it is long enough to be in its own way. The rule-based snake never steps into danger while a safe move exists. It dies only when all three cells around its head are blocked, usually by its own body and sometimes by its body together with a wall: it has sealed itself in.*
</details>

### Exercise 3 (The size of the board)
Measure the rule-based snake on boards of size 6, 10 and 20 (200 games each). Print the average score, and also the average score as a percentage of the number of cells. On which board does it fill the largest share?

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
for size in (6, 10, 20):
    average, best, length = measure(SnakeEnv(size=size), rule_based, games=200)
    print(f"{size:>2} x {size:<2}: average score {average:5.1f}, best {best:>3}, "
          f"that is {100 * average / size ** 2:4.1f}% of the cells")
```

*A larger board gives a larger score, because there is more room before the snake gets in its own way. As a share of the board the result is worse: the snake traps itself when it is still short compared with the space available. The cause is not a lack of room. It is that the rules cannot see where the body lies.*
</details>

### Exercise 4 (A world without walls)
Make a subclass `WrapSnakeEnv` in which the board wraps around: a snake leaving on the right re-enters on the left, and the same for top and bottom. Only the snake's own body is deadly. You need to change `_cell_ahead` (use the remainder `%`) and `_is_deadly`. Check it with `check_env` and measure the rule-based snake in it.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
class WrapSnakeEnv(SnakeEnv):
    def _cell_ahead(self, direction):
        dx, dy = self.DIRECTIONS[direction]
        return ((self.snake[0][0] + dx) % self.size, (self.snake[0][1] + dy) % self.size)

    def _is_deadly(self, cell):
        return cell in self.snake


check_env(WrapSnakeEnv(), skip_render_check=True)
for name, environment in (("with walls", SnakeEnv()), ("wrap-around", WrapSnakeEnv())):
    average, best, length = measure(environment, rule_based)
    print(f"{name:12s} average score {average:.1f}, best {best}")
```

*Two small methods changed, and `reset`, `step`, `render` and the observation all still work, because they were written in terms of those two helpers. The food features still mean "left of the head on the screen", which is no longer always the shortest way in a world that wraps around.*
</details>

### Exercise 5 (How many situations are there really?)
The observation has 11 bits, so at most 2048 values. Collect every different observation that occurs in 500 games of the rule-based snake and 500 games of the random snake, and count them. Then work out, by reasoning about which combinations of bits are possible, how many observations can occur at all.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
seen = set()
for policy in (rule_based, random_policy):
    for game in range(500):
        observation, info = env.reset(seed=game)
        terminated = truncated = False
        while not (terminated or truncated):
            seen.add(tuple(int(bit) for bit in observation))
            observation, reward, terminated, truncated, info = env.step(policy(observation))

print("different observations seen:", len(seen))
print("possible by reasoning:      ", 8 * 4 * 8)
```

*The three danger bits are free: 8 combinations. Exactly one of the four direction bits is true: 4 possibilities. The food is left, right or in the same column (3 cases) and above, below or in the same row (3 cases), but it cannot be in the same column and the same row as the head, which leaves 8. In total 8 × 4 × 8 = 256. A table with 256 rows and 3 columns will hold everything the agent can ever learn about this game.*
</details>

### Exercise 6 (Seeing the whole board (a little harder))
Write a function `board(env)` that returns the full situation as an array of shape `(3, size, size)`: one layer with a 1 at the head, one with 1s on the body, one with a 1 at the food. Play 200 random games and count how many **different boards** occur, and how many different 11-bit observations occur in the same games. What does the comparison tell you about learning with a table?

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
def board(env):
    layers = np.zeros((3, env.size, env.size), dtype=np.int8)
    head_x, head_y = env.snake[0]
    layers[0, head_y, head_x] = 1
    for x, y in env.snake[1:]:
        layers[1, y, x] = 1
    layers[2, env.food[1], env.food[0]] = 1
    return layers


boards, observations, steps = set(), set(), 0
for game in range(200):
    observation, info = env.reset(seed=game)
    terminated = truncated = False
    while not (terminated or truncated):
        boards.add(board(env).tobytes())
        observations.add(tuple(int(bit) for bit in observation))
        observation, reward, terminated, truncated, info = env.step(random_policy(observation))
        steps += 1

print(f"steps played:            {steps}")
print(f"different boards:        {len(boards)}")
print(f"different observations:  {len(observations)}")
```

*Almost every step produces a board that has never been seen before, so a table indexed by boards would learn nothing: each row would be updated once. The 11-bit observation maps thousands of boards to a hundred or so rows, and each row is visited many times. This is **generalisation by hand**. In Notebook 11 a neural network will take an array like `board(env)` as its input and find its own features.*
</details>

## 🔑 Key takeaways

- An environment is five decisions: **actions**, **observations**, **reward**, **end of episode**, **rendering**.
- The full state of even a small game is far too large for a table: a snake of length 11 on a 10 × 10 board has over 100 million states.
- A compact **observation** built from features makes tabular learning possible, at the price of **hiding information** from the agent.
- A good **reward** names the outcome wanted and leaves the method to the agent.
- Every episode needs a way to end: a **time limit** prevents endless safe loops.
- Measure **baselines** first. A random agent and a simple rule tell you what a learned agent has to beat.

---
**Next:** *Notebook 07: A Snake That Learns*, where Q-learning takes over from our hand-written rules.

---
*© Ioannis Tsioulis. *From Random Moves to Winning Agents: Reinforcement Learning with Games*. Prepared for educational use.*